In [162]:
#MLP Paper

import torch 
import torch.nn.functional as F
import matplotlib.pyplot as plt #Making features
%matplotlib inline


In [163]:
words = open('names.txt','r').read().splitlines()
words[:8]


['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [164]:
len(words)

32033

In [165]:
chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i , s in enumerate(chars)}
stoi['.']=0
itos ={i:s for s , i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [166]:
# Make DATA SET 
block_size =3# context length how many chars do we take to predict the next one?
X , Y =[] ,[]
for w in words:
    #print(w)
    context = [0] * block_size
    for ch in w + '.':
        ix = stoi[ch]
        X.append(context)
        Y.append(ix)
        #print(''.join(itos[i] for i in context), '--->', itos[ix])
        context = context[1:] + [ix]

X = torch.tensor(X)
Y = torch.tensor(Y)        

In [167]:
X.shape , X.dtype , Y.shape , Y.dtype

(torch.Size([228146, 3]), torch.int64, torch.Size([228146]), torch.int64)

In [168]:
emb = C[X]
emb.shape

torch.Size([228146, 3, 2])

In [169]:
W1 = torch.randn((6,100))
b1 = torch.randn(100)

In [170]:
h = torch.tanh(emb.view(-1,6) @ W1 + b1) # -1 show discover what it's !!

In [171]:
##torch.cat(torch.unbind(emb,1),1).shape emb.view(32,6) == torch.cat(torch.unbind(emb,1),1)

In [172]:
h.shape


torch.Size([228146, 100])

In [173]:
W2 = torch.randn((100,27))
b2 = torch.randn(27)


In [174]:
logits = h@ W2 + b2
logits.shape

torch.Size([228146, 27])

In [175]:
counts = logits.exp()

In [176]:
prob = counts / counts.sum(1,keepdim=True)

In [177]:
prob.shape

torch.Size([228146, 27])

In [178]:
#loss = -prob[torch.arange(32),Y].log().mean()
#loss

In [179]:
#====Woops===#
g = torch.Generator().manual_seed(2147483647)
C=torch.randn((27,2),generator=g)
W1 = torch.randn((6,100),generator=g)
b1 = torch.randn(100,generator=g)
W2 = torch.randn((100,27),generator=g)
b2 = torch.randn(27,generator=g)
parameters = [C,W1,b1,W2,b2]

In [180]:
sum(p.nelement()for p in parameters)

3481

In [181]:
for p in parameters:
    p.requires_grad = True

In [ ]:
for _ in range(1000):
    ix = torch.randint(0,X.shape[0],(32,))
     
#FORWARD PASS
    emb = C[X[ix]] # (32,3,2)
    h = torch.tanh(emb.view(-1,6)@W1+b1) #(32,100)
    logits = h @ W2 + b2 # (32,27)
        #counts = logits.exp()
        #prob = counts / counts.sum(1,keepdims =True)
        #loss = -prob[torch.arange(32), Y].log().mean() All comprssed in one line
    loss = F.cross_entropy(logits,Y[ix])
    #print(loss.item())
    #Backward
    for p in parameters:
        p.grad = None
    loss.backward()
    #backward
    for p in parameters:
        p.data += -0.1 * p.grad    
print(loss.item())

2.578265428543091
